# Avito — кандидатогенерация, пайплайн v2

## Что делает решение

Ноутбук реализует этап кандидатогенерации для поиска услуг Avito: по поисковому запросу и сопутствующим признакам формируется пул до 50 `item_id`, который затем передаётся на ранжирование. Главная метрика этого этапа — **Recall@50**, поэтому архитектура оптимизируется прежде всего под полноту: объявление, не попавшее в кандидаты, уже не может быть восстановлено реранкером.

## Почему выбран именно такой пайплайн

**1. Сначала проверяем структуру данных, а не строим модель на предположениях.**  
В обучающей выборке около 497 тыс. строк, после дедупликации остаётся около 467 тыс. кликов; уникальных текстов запросов — около 74 тыс., а запросов-единиц с учётом локации, категории, фильтров и доставки — около 354 тыс. Медиана числа кликов на запрос-единицу равна 1. Это означает, что история поведения разрежена, поэтому одного поведенческого сигнала недостаточно.

Особенно важно, что из более чем 344 тыс. уникальных обучающих объявлений только около 18 тыс. присутствуют в `benchmark_items` (примерно 5.3%). Поэтому нельзя строить решение вокруг точного совпадения `item_id`: большая часть релевантных объявлений на инференсе для обучения напрямую недоступна. Выгоднее дообучить семантическое представление по текстам объявлений и переносить знания через смысловую близость.

**2. Семантический канал — базовый источник устойчивости к перефразированию.**  
Используется `intfloat/multilingual-e5-small`, дообученная на парах «запрос → кликнутое объявление» с контрастивной потерей InfoNCE. Негативы берутся внутри батча, а совпадающие запросы/объявления маскируются, чтобы повторяющиеся пары не становились ложными негативами.

Практический плюс — доменная адаптация без ручной разметки: модель учится различать близкие по словам, но разные по смыслу услуги. Это особенно полезно там, где лексическое совпадение само по себе недостаточно.

При этом часть запросов и их объявления намеренно откладывается до обучения реранкера. Такое разделение уменьшает риск утечки разметки и делает валидацию ближе к реальному инференсу.

**3. Лексический канал нужен как страховка для точных терминов.**  
BM25 реализован через `scipy.sparse`, а не построчный `rank_bm25`. На большом корпусе векторные операции дают существенно меньшую вычислительную стоимость, а результат остаётся лексически интерпретируемым: точные слова и редкие токены получают естественный приоритет.

В индексах отдельно учитываются заголовок и заголовок вместе с параметрами. Полное описание не включается в лексический канал: оно значительно длиннее и шумнее, а его смысловое содержание уже покрывает E5. Дополнительно вычисляются признаки покрытия токенов запроса — они полезны реранкеру как сигнал точного совпадения.

**4. Поведенческий канал `knn` дополняет семантический поиск.**  
Для нового запроса находятся близкие обучающие запросы, после чего по их кликам строится взвешенный псевдо-документ. Такой механизм переносит поведение пользователей даже тогда, когда точного `item_id` в целевом корпусе нет. Это выгоднее прямого history-поиска, потому что работает и при холодном старте.

Для точного совпадения текста отдельно используется `hist`: если по тому же запросу кликнуто объявление, которое действительно есть в бенчмарке, оно получает приоритет. Этот сигнал ограничен небольшим числом объявлений, поэтому он усиливает надёжный случай, не вытесняя остальные каналы.

**5. Локация используется мягко, а не как жёсткий фильтр.**  
В данных около 82.2% кликов имеют совпадающую локацию поиска и объявления. Поэтому отдельный локационный канал оправдан. Но жёстко отбрасывать другие локации рискованно: ошибка классификации, неполное пересечение категорий или особенности доставки могут сделать релевантное объявление «чужим».

Поэтому локация ограничивает отдельные каналы, но общий поиск всё равно сохраняется. При поиске с доставкой локационный сигнал отключается, потому что географическая близость в этом сценарии менее показательна.

**6. Каналы объединяются через Reciprocal Rank Fusion.**  
Dense, BM25 и поведенческий поиск дают оценки в разных шкалах. Прямое смешивание сырых score потребовало бы отдельной калибровки и могло бы нестабильно работать на разных запросах. RRF объединяет именно ранги, поэтому не зависит от масштаба исходных оценок.

Веса каналов и `k_rrf` подбираются на отложенной части данных по целевой функции, учитывающей и Recall@50, и полноту пула кандидатов `k_cand`. Это соответствует каскадной природе системы: важно не только вернуть релевантное объявление в финальные 50, но и не потерять его до реранжирования.

**7. Реранкер CatBoost применяется только после широкого retrieval.**  
CatBoost с `YetiRank` получает признаки всех каналов, совпадений категории/локации, географии, рейтинга, отзывов, популярности и покрытия токенов. Он работает на тех же кандидатах, которые формируются при инференсе, поэтому обучение соответствует реальному распределению входных объектов.

Размер пула ограничен `k_cand = 300`: этого достаточно, чтобы реранкер имел запас для перестановки кандидатов, но не приходилось считать дорогие признаки по всему корпусу. При этом финальная выдача ограничивается `final_k = 50`, как требует метрика.

## Почему отвергнуты более простые альтернативы

**Один поисковый канал.** Лексический поиск хорошо ловит точные термины, но слабее при перефразировании; семантический устойчив к смысловым вариациям, но может пропустить редкие слова; поведенческий канал зависит от близости к истории. Комбинация каналов снижает риск провала на разных типах запросов.

**Жёсткая фильтрация по категории.** Пересечение категорий train и benchmark составляет только 3 из 47, поэтому жёсткое отсечение может удалить релевантные объявления из-за различий в таксономии. Категория выгоднее как мягкий признак и дополнительный канал.

**Полные матрицы `query × corpus`.** Они резко увеличивают память и время. Поэтому поиск делается батчами, а признаки считаются только для уже отобранного пула кандидатов.

**Cross-encoder как отдельный этап.** Для Recall@50 критично само попадание объявления в пул. Cross-encoder дорог по вычислениям, поэтому в текущей конфигурации его стоимость не оправдывает использование раньше реранкера; при наличии дополнительного бюджета его разумнее добавить как признак.

**Отдельная модель на каждую категорию.** Распределение категорий сильно неравномерно: одна категория занимает более 99% данных, а остальные представлены очень небольшими объёмами. Одна модель с категориальными признаками использует общую статистическую базу и не дробит дефицитную разметку на малые подвыборки.

## Почему решения выгодны по вычислительной стоимости

Основные узкие места устранены архитектурно: BM25 считается векторно через sparse-матрицы, тексты кодируются батчами с сортировкой по длине, эмбеддинги и дообученная модель кэшируются, а score/признаки не пересчитываются для всего `query × corpus`. Это уменьшает и время, и пиковое потребление памяти.

При исчерпании бюджета предусмотрен практический fallback: можно использовать результаты RRF без реранкера и всё равно сформировать валидный `answer.csv`. Таким образом, система не зависит от успешного завершения самого дорогого этапа.

## Воспроизводимость и ожидаемый результат

Все ключевые компоненты запускаются локально без внешних API: `transformers`, `scipy`, `scikit-learn`, `catboost`, `optuna`, `nltk`. Эмбеддинги и дообученная модель сохраняются в рабочую директорию, поэтому повторный запуск может использовать кэш.

В исходном описании этой конфигурации зафиксирован **Recall@50 = 0.88**. Итоговый файл должен содержать ровно `query_id` и `answer`; в `answer` — не более 50 уникальных `item_id`, существующих в `benchmark_items`.

**Важно для загрузки данных:** данные читаются непосредственно из Kaggle Dataset `baldmonkeyuuu/avito-dataset` через `KaggleDatasetAdapter.PANDAS`. Это снимает зависимость от жёстко заданных `/kaggle/input/...` путей и позволяет ноутбуку получать актуальную версию датасета через стандартный KaggleHub-интерфейс. Остальная логика пайплайна использует те же три DataFrame (`train_raw`, `bq_raw`, `bi_raw`), поэтому менять её ради нового способа загрузки не требуется.


In [25]:
# ЯЧЕЙКА 1: зависимости (torch / transformers / sklearn / scipy на Kaggle уже есть)
!pip install -q catboost optuna nltk pyarrow


In [26]:
# ЯЧЕЙКА 2: импорты и CONFIG
import os, re, gc, time, math, random
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm

T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.0f}s] {msg}", flush=True)


try:
    import psutil
    def mem():
        return psutil.Process().memory_info().rss / 1e9
except ImportError:
    def mem():
        return float("nan")

CFG = {
    # --- пути ---
    "train_path":         "train.parquet",
    "bench_queries_path": "benchmark_queries.parquet",
    "bench_items_path":   "benchmark_items.parquet",
    "work_dir":   ".",
    "output_csv": "answer.csv",
    "use_cache":  True,          # кэш эмбеддингов и дообученной модели в work_dir

    # --- энкодер ---
    "model_name":   "intfloat/multilingual-e5-small",
    "max_len_doc":  256,         # заголовок+параметры идут первыми, длинное описание обрезается
    "max_len_q":    64,
    "enc_bs":       512,
    # --- дообучение (in-batch negatives). ft_epochs=0 -> без дообучения ---
    "ft_epochs":    5,
    "ft_bs":        256,         # при OOM уменьшить до 128
    "ft_lr":        3e-5,
    "ft_temp":      0.05,
    "ft_max_pairs": None,        # для отладки можно ограничить, напр. 50000

    # --- разбиение train ---
    "holdout_frac":   0.25,      # доля запросов (по тексту), которые НЕ идут в дообучение и учат реранкер
    "rank_max_units": 24000,     # сколько «запросов-единиц» держим для реранкера (ограничение по времени)

    # --- ретривал ---
    "knn_m":     30,             # соседей среди train-запросов для канала knn (query expansion по кликам)
    "knn_temp":  0.05,
    "k_ch":      300,            # top-k в каждом канале
    "k_cand":    300,            # сколько кандидатов после RRF идёт в реранкер
    "final_k":   50,
    "hist_max":  15,             # макс. объявлений из точного history-совпадения (только если они есть в корпусе)
    "loc_channel": "auto",       # "auto" | True | False — канал «только объявления из той же локации»
    "loc_min_share": 0.30,       # порог для auto: доля train-кликов, где item_location == search_location
    "rrf_trials": 60,
    "rrf_tune_units": 5000,

    # --- реранкер ---
    "cb_iterations": 1500,
    "cb_lr": 0.08,
    "cb_depth": 8,
    "shap_rows": 20000,

    "seed": 42,
}
np.random.seed(CFG["seed"]); random.seed(CFG["seed"])
log(f"импорты ok, RSS={mem():.1f} GB")


[      0s] импорты ok, RSS=18.8 GB


In [27]:
# ЯЧЕЙКА 3: токенизация (Snowball-стемминг + кэш), нормализация, метрика
from nltk.stem.snowball import SnowballStemmer

_stem = SnowballStemmer("russian")
_stem_cache = {}
_TOKEN_RE = re.compile(r"[а-яеa-z0-9]+")


def tokenize(text):
    """Токены + стемминг. 'ё' -> 'е', чтобы 'ёлка' и 'елка' совпадали."""
    if not isinstance(text, str) or not text:
        return []
    out = []
    for t in _TOKEN_RE.findall(text.lower().replace("ё", "е")):
        if len(t) <= 1:
            continue
        s = _stem_cache.get(t)
        if s is None:
            s = _stem.stem(t)
            _stem_cache[t] = s
        out.append(s)
    return out


def _identity(x):
    return x


def norm_q(s):
    """Нормализация текста запроса — ОДНА функция и для train, и для бенчмарка."""
    return re.sub(r"\s+", " ", str(s).lower().replace("ё", "е")).strip()


def _idstr(s: pd.Series) -> pd.Series:
    """Идентификаторы -> строки одного формата (без '123.0' у float-колонок)."""
    if pd.api.types.is_float_dtype(s):
        try:
            s = s.astype("Int64")
        except Exception:
            pass
    return s.astype("string").fillna("NA").astype(str)


def haversine_km(lat1, lon1, lat2, lon2):
    p = np.pi / 180.0
    a = (np.sin((lat2 - lat1) * p / 2) ** 2
         + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2)
    return 12742.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


In [28]:
# ЯЧЕЙКА 4: препроцессинг
ITEM_TEXT = ["item_title_raw", "item_description_raw", "item_infm_params_text"]
ITEM_ID_COLS = ["item_category_id", "item_microcat_id", "item_location_id"]
ITEM_NUM = ["item_price", "item_rating", "item_rating_reviews_count", "item_latitude",
            "item_longitude", "item_is_phone_hidden", "item_is_message_forbidden"]
UNIT_KEYS = ["q_norm", "search_location_id", "search_category",
             "search_infm_params_text", "search_is_delivery_search"]


def prep_items(df: pd.DataFrame) -> pd.DataFrame:
    """Единый формат таблицы объявлений (и для корпуса бенчмарка, и для train-объявлений)."""
    df = df.reset_index(drop=True)
    out = pd.DataFrame({"item_id": df["item_id"].astype(str)})
    for c in ITEM_TEXT:
        out[c] = df[c].fillna("").astype(str) if c in df else ""
    for c in ITEM_ID_COLS:
        out[c] = _idstr(df[c]) if c in df else "NA"
    for c in ITEM_NUM:
        out[c] = (pd.to_numeric(df[c], errors="coerce").astype("float32")
                  if c in df else np.float32(np.nan))
    return out


def prep_queries(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    out = pd.DataFrame({"q_norm": df["search_query"].fillna("").map(norm_q)})
    out["search_location_id"] = _idstr(df["search_location_id"]) if "search_location_id" in df else "NA"
    out["search_category"] = _idstr(df["search_category"]) if "search_category" in df else "NA"
    out["search_infm_params_text"] = (df["search_infm_params_text"].fillna("").astype(str)
                                      if "search_infm_params_text" in df else "")
    out["search_is_delivery_search"] = (pd.to_numeric(df["search_is_delivery_search"], errors="coerce")
                                        .fillna(0).astype(int) if "search_is_delivery_search" in df else 0)
    if "query_id" in df:
        out["query_id"] = df["query_id"].astype(str)
    return out


def prep_train(train_raw: pd.DataFrame):
    """-> pairs (одна строка = клик; поля запроса + id и признаки объявления), train_items (уникальные)."""
    q = prep_queries(train_raw)
    items_all = prep_items(train_raw)
    pairs = pd.concat([q, items_all[["item_id", "item_category_id", "item_location_id",
                                     "item_latitude", "item_longitude"]]], axis=1)
    # ВАЖНО: «запрос» = связка (текст, локация, категория, фильтры, доставка) — аналог query_id бенчмарка.
    pairs["uid"] = pairs.groupby(UNIT_KEYS, sort=False).ngroup().astype(np.int64)
    train_items = items_all.drop_duplicates("item_id").reset_index(drop=True)
    pairs = pairs.drop_duplicates(["uid", "item_id"]).reset_index(drop=True)
    return pairs, train_items


def doc_text(items: pd.DataFrame) -> pd.Series:
    """Текст объявления для энкодера: заголовок, параметры, описание (в таком порядке — из-за обрезки)."""
    t = items["item_title_raw"] + ". " + items["item_infm_params_text"] + ". " + items["item_description_raw"]
    return t.str.slice(0, 1200)


class Vocab:
    """Единое отображение строковых id -> int (категории, микрокатегории, локации)."""
    def __init__(self, *series):
        u = pd.unique(pd.concat([pd.Series(s).astype(str) for s in series], ignore_index=True))
        self.map = {v: i for i, v in enumerate(u)}
        self.n = len(u)

    def code(self, s):
        return pd.Series(s).astype(str).map(self.map).fillna(-1).astype(np.int32).values


In [29]:
# ЯЧЕЙКА 5: ДИАГНОСТИКА — проверяем допущения о данных (то, что v1 принимал на веру)
def diagnostics(pairs, train_items, bench_q, bench_i):
    log("=" * 70)
    log("ДИАГНОСТИКА ДАННЫХ")
    ns = pairs["uid"].nunique()
    print(f"строк(кликов)={len(pairs)}, уникальных текстов={pairs.q_norm.nunique()}, "
          f"уникальных «запросов-единиц»={ns}, кликов на единицу: mean={len(pairs)/ns:.2f}, "
          f"median={pairs.groupby('uid').size().median():.0f}")
    inter = len(set(train_items.item_id) & set(bench_i.item_id))
    print(f"train-объявлений всего={len(train_items)}; из них есть в корпусе бенчмарка: {inter} "
          f"({inter/len(train_items):.1%}). Если ~0% — history по item_id почти бесполезен, "
          f"а train нужен для обучения по ТЕКСТАМ объявлений.")
    seen = bench_q.q_norm.isin(set(pairs.q_norm)).mean()
    print(f"доля запросов бенчмарка, чей текст встречается в train: {seen:.1%}")
    print("\nsearch_category в train (топ):", pairs.search_category.value_counts().head(6).to_dict())
    print("search_category в бенчмарке (топ):", bench_q.search_category.value_counts().head(6).to_dict())
    print("item_category_id в корпусе: unique =", bench_i.item_category_id.nunique(),
          "| топ:", bench_i.item_category_id.value_counts().head(6).to_dict())
    print("кросс-таб train: search_category -> item_category_id (топ-3 на строку):")
    ct = pd.crosstab(pairs.search_category, pairs.item_category_id)
    for cat, row in ct.iterrows():
        print(f"   {cat}: {row.sort_values(ascending=False).head(3).to_dict()}")
    cat_eq = (pairs.search_category == pairs.item_category_id).mean()
    loc_eq = (pairs.search_location_id == pairs.item_location_id).mean()
    print(f"\nдоля кликов с item_category_id == search_category: {cat_eq:.1%}")
    print(f"доля кликов с item_location_id == search_location_id: {loc_eq:.1%}   <- решает, включать ли канал локации")
    print(f"доставка (search_is_delivery_search=1) в train: {(pairs.search_is_delivery_search==1).mean():.1%}")
    print("=" * 70)
    return loc_eq


In [30]:
# ЯЧЕЙКА 6: энкодер (E5) и дообучение на кликах
class Encoder:
    """Обёртка над HF-энкодером: mean pooling + L2-нормализация, как у E5."""
    def __init__(self, path, device=None):
        import torch
        from transformers import AutoTokenizer, AutoModel
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.use_amp = self.device.startswith("cuda")
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModel.from_pretrained(path).to(self.device).eval()
        log(f"[encoder] {path} on {self.device}")

    def embed(self, texts, max_len):
        torch = self.torch
        b = self.tok(texts, padding=True, truncation=True, max_length=max_len,
                     return_tensors="pt").to(self.device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp):
            h = self.model(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
        e = (h * m).sum(1) / m.sum(1).clamp(min=1)
        return torch.nn.functional.normalize(e.float(), dim=-1)

    def encode(self, texts, prefix, max_len, bs=512, desc="encode"):
        """Батчи сортируются по длине -> минимум паддинга (v1 кодировал 38 мин из-за max_len=512 без сортировки)."""
        texts = [prefix + t for t in texts]
        n = len(texts)
        order = np.argsort([len(t) for t in texts])
        out = None
        with self.torch.inference_mode():
            for s in tqdm(range(0, n, bs), desc=desc):
                ids = order[s:s + bs]
                e = self.embed([texts[i] for i in ids], max_len).cpu().numpy()
                if out is None:
                    out = np.empty((n, e.shape[1]), dtype=np.float32)
                out[ids] = e
        return out

    def save(self, path):
        os.makedirs(path, exist_ok=True)
        self.model.save_pretrained(path)
        self.tok.save_pretrained(path)


Q_PREFIX, D_PREFIX = "query: ", "passage: "


def finetune(enc: Encoder, q_texts, d_texts, cfg):
    """Контрастивное дообучение (InfoNCE, in-batch negatives) на парах «запрос -> кликнутое объявление».
    Ложные негативы (тот же запрос или то же объявление в батче) маскируются."""
    torch = enc.torch
    n = len(q_texts)
    qid = pd.factorize(pd.Series(q_texts))[0]
    did = pd.factorize(pd.Series(d_texts))[0]
    bs, ep = cfg["ft_bs"], cfg["ft_epochs"]
    steps_per_ep = n // bs
    total = steps_per_ep * ep
    warm = max(1, int(0.05 * total))
    opt = torch.optim.AdamW(enc.model.parameters(), lr=cfg["ft_lr"], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=enc.use_amp)
    rng = np.random.RandomState(cfg["seed"])
    enc.model.train()
    step = 0
    for e in range(ep):
        perm = rng.permutation(n)
        pbar = tqdm(range(steps_per_ep), desc=f"finetune epoch {e+1}/{ep}")
        run = 0.0
        for i in pbar:
            idx = perm[i * bs:(i + 1) * bs]
            eq = enc.embed([Q_PREFIX + q_texts[j] for j in idx], cfg["max_len_q"])
            ed = enc.embed([D_PREFIX + d_texts[j] for j in idx], cfg["max_len_doc"])
            logits = (eq @ ed.T) / cfg["ft_temp"]
            qb = torch.as_tensor(qid[idx], device=logits.device)
            db = torch.as_tensor(did[idx], device=logits.device)
            eye = torch.eye(len(idx), dtype=torch.bool, device=logits.device)
            fneg = ((qb[:, None] == qb[None, :]) | (db[:, None] == db[None, :])) & ~eye
            logits = logits.masked_fill(fneg, -1e4)
            lab = torch.arange(len(idx), device=logits.device)
            ce = torch.nn.functional.cross_entropy
            loss = 0.5 * (ce(logits, lab) + ce(logits.T, lab))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(enc.model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
            run = 0.98 * run + 0.02 * loss.item() if i else loss.item()
            if i % 50 == 0:
                pbar.set_postfix(loss=f"{run:.3f}")
    enc.model.eval()
    return enc


In [31]:
# ЯЧЕЙКА 7: индексы. BM25 на scipy.sparse (вместо rank_bm25: тот считает get_scores питоновскими циклами)
def bm25_matrix(X, k1=1.2, b=0.75):
    """Матрица весов BM25 (docs x vocab): вес(term, doc) уже включает idf, tf-насыщение и длину документа."""
    X = X.tocsr().astype(np.float32)
    N = X.shape[0]
    dl = np.asarray(X.sum(1)).ravel()
    avg = max(float(dl.mean()), 1e-6)
    df = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.log(1.0 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    tf = X.data
    data = idf[X.indices] * tf * (k1 + 1.0) / (tf + k1 * (1.0 - b + b * dl[rows] / avg))
    return sp.csr_matrix((data.astype(np.float32), X.indices, X.indptr), shape=X.shape)


class Corpus:
    """Набор объявлений, по которому ищем: BM25 по двум полям, эмбеддинги, метаданные (всё в numpy)."""
    def __init__(self, items: pd.DataFrame, emb: np.ndarray, voc: dict, name="corpus"):
        t = time.time()
        self.name, self.items, self.n = name, items, len(items)
        self.emb = np.ascontiguousarray(emb, dtype=np.float32)
        assert len(self.emb) == self.n
        self.ids = items["item_id"].values
        self.id2pos = {i: p for p, i in enumerate(self.ids)}

        tt = [tokenize(x) for x in items["item_title_raw"].values]
        tp = [tokenize(x) for x in items["item_infm_params_text"].values]
        td = [tokenize(x) for x in items["item_description_raw"].values]
        tokA = [a + b for a, b in zip(tt, tp)]            # поле A: заголовок + параметры
        tokF = [a + d for a, d in zip(tokA, td)]          # поле F: всё вместе, с описанием
        self.cv = CountVectorizer(analyzer=_identity, dtype=np.float32)
        XF = self.cv.fit_transform(tokF)
        XA, XP = self.cv.transform(tokA), self.cv.transform(tp)
        self.WA = bm25_matrix(XA).T.tocsr()               # vocab x docs
        self.WF = bm25_matrix(XF).T.tocsr()
        self.BA = (XA > 0).astype(np.float32).T.tocsr()   # бинарные — для доли покрытия токенов запроса
        self.BF = (XF > 0).astype(np.float32).T.tocsr()
        self.BP = (XP > 0).astype(np.float32).T.tocsr()   # только параметры (для сравнения с фильтрами)
        dfp = np.asarray((XP > 0).sum(0)).ravel()
        self.idf_p = (np.log((self.n + 1.0) / (dfp + 1.0)) + 1.0).astype(np.float32)
        self.title_ntok = np.array([len(x) for x in tt], np.float32)
        self.desc_ntok = np.array([len(x) for x in td], np.float32)
        del tt, tp, td, tokA, tokF, XF, XA, XP

        it = items
        price = it["item_price"].values.astype(np.float32)
        self.price_log = np.log1p(np.clip(price, 0, None))            # NaN сохраняем: CatBoost их понимает
        self.rating = it["item_rating"].values.astype(np.float32)
        self.reviews_log = np.log1p(np.clip(np.nan_to_num(it["item_rating_reviews_count"].values), 0, None)).astype(np.float32)
        self.lat = it["item_latitude"].values.astype(np.float32)
        self.lon = it["item_longitude"].values.astype(np.float32)
        self.phone_hidden = it["item_is_phone_hidden"].values.astype(np.float32)
        self.msg_forbidden = it["item_is_message_forbidden"].values.astype(np.float32)
        self.cat_code = voc["cat"].code(it["item_category_id"])
        self.mc_code = voc["mc"].code(it["item_microcat_id"])
        self.loc_code = voc["loc"].code(it["item_location_id"])
        log(f"[{name}] корпус: {self.n} объявлений, словарь {len(self.cv.vocabulary_)}, "
            f"{time.time()-t:.0f}с, RSS={mem():.1f} GB")


def _parse_rating(x):
    m = re.search(r"рейтинг[^\d]{0,40}(\d(?:[.,]\d)?)", x.lower().replace("ё", "е"))
    return float(m.group(1).replace(",", ".")) if m else np.nan


def _strip_rating(x):
    return re.sub(r"рейтинг[^;\n|]*", " ", x.lower())


class QSet:
    """Набор запросов (строки benchmark_queries или «запросы-единицы» train) в виде numpy-массивов."""
    def __init__(self, q_df: pd.DataFrame, emb, voc, loc_geo):
        d = q_df.reset_index(drop=True)
        self.df, self.n, self.emb = d, len(d), np.ascontiguousarray(emb, dtype=np.float32)
        self.loc_code = voc["loc"].code(d["search_location_id"])
        self.cat_code = voc["cat"].code(d["search_category"])
        self.delivery = d["search_is_delivery_search"].values.astype(int)
        self.tok = [tokenize(t) for t in d["q_norm"].values]
        self.ntok = np.array([max(len(set(t)), 1) for t in self.tok], np.float32)
        self.nchar = d["q_norm"].str.len().values.astype(np.float32)
        ft = d["search_infm_params_text"].values
        self.has_filter = np.array([bool(x.strip()) for x in ft], np.float32)
        self.rating_thr = np.array([_parse_rating(x) for x in ft], np.float32)
        self.ftok = [tokenize(_strip_rating(x)) for x in ft]
        g = [loc_geo.get(x, (np.nan, np.nan)) for x in d["search_location_id"].values]
        self.lat = np.array([a for a, _ in g], np.float32)
        self.lon = np.array([b for _, b in g], np.float32)
        self._bound = None

    def bind(self, corpus: Corpus):
        """Матрицы запросов в пространстве словаря корпуса (кэшируются)."""
        if self._bound is not None and self._bound[0] is corpus:
            return self._bound[1:]
        Qb = (corpus.cv.transform(self.tok) > 0).astype(np.float32).tocsr()
        Fq = (corpus.cv.transform(self.ftok) > 0).astype(np.float32).tocsr()
        Fw = Fq.multiply(corpus.idf_p[None, :]).tocsr()
        fden = np.asarray(Fw.sum(1)).ravel().astype(np.float32)
        self._bound = (corpus, Qb, Fw, fden)
        return Qb, Fw, fden


class QueryPool:
    """Пул train-запросов с их кликами. Для нового запроса берём m ближайших train-запросов и усредняем
    ЭМБЕДДИНГИ кликнутых ими объявлений -> «псевдо-объявление» для поиска (канал knn), а также
    распределение микрокатегорий/категорий (признаки реранкера). Работает, даже если train-объявлений нет в корпусе."""
    def __init__(self, pairs, text2row, q_emb_all, tid2row, item_emb, item_mc, item_cat, n_mc, n_cat):
        texts = pairs["q_norm"].unique()
        t_idx = pd.Series(np.arange(len(texts)), index=texts)
        r = pairs["q_norm"].map(t_idx).values
        c = pairs["item_id"].map(tid2row).values
        ok = ~pd.isna(c)
        r, c = r[ok].astype(np.int64), c[ok].astype(np.int64)
        n_items = len(item_emb)
        M = sp.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(len(texts), n_items))
        M.sum_duplicates(); M.data[:] = 1.0
        deg = np.asarray(M.sum(1)).ravel()
        Mn = sp.diags(1.0 / np.maximum(deg, 1.0)).astype(np.float32) @ M
        C = np.asarray(Mn @ item_emb, dtype=np.float32)
        self.C = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)
        self.Q = np.ascontiguousarray(q_emb_all[[text2row[t] for t in texts]], dtype=np.float32)

        def onehot(code, n):
            v = code >= 0
            return sp.csr_matrix((np.ones(v.sum(), np.float32), (np.where(v)[0], code[v])), shape=(len(code), n))
        self.Mc = (Mn @ onehot(item_mc, n_mc)).tocsr()
        self.Cc = (Mn @ onehot(item_cat, n_cat)).tocsr()
        self.texts = set(texts)
        log(f"[pool] {len(texts)} train-запросов, {len(r)} кликов")

    def expand(self, qe, m, temp):
        S = qe @ self.Q.T
        m = min(m, S.shape[1])
        idx = np.argpartition(S, -m, axis=1)[:, -m:]
        sims = np.take_along_axis(S, idx, 1)
        w = np.exp((sims - sims.max(1, keepdims=True)) / temp)
        w /= w.sum(1, keepdims=True)
        pseudo = np.einsum("qm,qmd->qd", w.astype(np.float32), self.C[idx])
        pseudo /= np.linalg.norm(pseudo, axis=1, keepdims=True) + 1e-9
        W = sp.csr_matrix((w.ravel(), (np.repeat(np.arange(len(qe)), m), idx.ravel())), shape=S.shape)
        return pseudo.astype(np.float32), sims.max(1), (W @ self.Mc).toarray(), (W @ self.Cc).toarray()


In [32]:
# ЯЧЕЙКА 8: мультиканальный ретривер (батчами по запросам), RRF и метрики
CHANNELS = [  # (имя, базовый скор, только-локация?)
    ("dense", "dn", False), ("knn", "kn", False), ("bm25A", "bA", False), ("bm25F", "bF", False),
    ("dense_loc", "dn", True), ("knn_loc", "kn", True), ("bm25A_loc", "bA", True),
    ("hist", None, False),
]
HIST = len(CHANNELS) - 1


def topk_rows(S, k):
    k = min(k, S.shape[1])
    idx = np.argpartition(S, -k, axis=1)[:, -k:]
    vals = np.take_along_axis(S, idx, 1)
    o = np.argsort(-vals, axis=1)
    return np.take_along_axis(idx, o, 1).astype(np.int32), np.take_along_axis(vals, o, 1)


class Retriever:
    def __init__(self, corpus: Corpus, pool: QueryPool, cfg: dict, use_loc: bool):
        self.c, self.pool, self.cfg, self.use_loc = corpus, pool, cfg, use_loc

    def _chunk(self, qs: QSet, a, b, feats=False):
        c = self.c
        Qb, Fw, fden = qs.bind(c)
        qe = qs.emb[a:b]
        pseudo, nn1, mcp, ctp = self.pool.expand(qe, self.cfg["knn_m"], self.cfg["knn_temp"])
        S = {"dn": qe @ c.emb.T, "kn": pseudo @ c.emb.T,
             "bA": (Qb[a:b] @ c.WA).toarray(), "bF": (Qb[a:b] @ c.WF).toarray(),
             "nn1": nn1, "mcp": mcp, "ctp": ctp}
        if feats:
            S["covA"] = (Qb[a:b] @ c.BA).toarray()
            S["covF"] = (Qb[a:b] @ c.BF).toarray()
            S["fcov"] = (Fw[a:b] @ c.BP).toarray()
        return S

    def channel_tops(self, qs: QSet, hist_lists=None, chunk=128):
        """Для каждого запроса: top-k позиций корпуса в каждом канале. -> tops (nq,C,K), valid (nq,C,K)."""
        K, nq, C = self.cfg["k_ch"], qs.n, len(CHANNELS)
        K = min(K, self.c.n)
        tops = np.zeros((nq, C, K), np.int32)
        valid = np.zeros((nq, C, K), bool)
        for a in tqdm(range(0, nq, chunk), desc="каналы"):
            b = min(nq, a + chunk)
            S = self._chunk(qs, a, b)
            for ci, (name, base, local) in enumerate(CHANNELS):
                if base is None:
                    continue
                M = S[base]
                if local:
                    if not self.use_loc:
                        continue
                    mask = self.c.loc_code[None, :] == qs.loc_code[a:b, None]
                    rowok = (qs.delivery[a:b] == 0) & (qs.loc_code[a:b] >= 0)
                    M = np.where(mask, M, -np.inf)
                idx, vals = topk_rows(M, K)
                v = np.isfinite(vals) & (vals > 0 if base in ("bA", "bF") else True)
                if local:
                    v &= rowok[:, None]
                tops[a:b, ci], valid[a:b, ci] = idx, v
            del S
        if hist_lists is not None:
            for i, h in enumerate(hist_lists):
                if len(h):
                    h = np.asarray(h[:K], np.int32)
                    tops[i, HIST, :len(h)], valid[i, HIST, :len(h)] = h, True
        return tops, valid

    def features(self, qs: QSet, cand, rrf, hist_keys=None, chunk=128):
        """Признаки пары (запрос, кандидат). Скоры всех каналов считаются для КАЖДОГО кандидата
        (а не только там, где он попал в top-k канала) — нет «дыр» вместо признаков."""
        c, nq, K = self.c, qs.n, cand.shape[1]
        parts = []
        for a in tqdm(range(0, nq, chunk), desc="признаки"):
            b = min(nq, a + chunk)
            S = self._chunk(qs, a, b, feats=True)
            cd = cand[a:b]
            ok = cd >= 0
            cs = np.where(ok, cd, 0)
            g = lambda M: np.take_along_axis(M, cs, 1)
            rowmax = lambda M: M.max(1, keepdims=True)
            qcol = lambda v: np.broadcast_to(v[a:b, None], cs.shape)
            dn, kn, bA, bF = g(S["dn"]), g(S["kn"]), g(S["bA"]), g(S["bF"])
            fden = qs.bind(c)[2][a:b, None]
            fc = np.where(fden > 0, g(S["fcov"]) / np.maximum(fden, 1e-6), np.nan)
            thr = qcol(qs.rating_thr)
            geo = haversine_km(qcol(qs.lat), qcol(qs.lon), c.lat[cs], c.lon[cs])
            lm = (c.loc_code[cs] == qcol(qs.loc_code)) & (qcol(qs.loc_code) >= 0)
            f = {
                "qrow": np.broadcast_to(np.arange(a, b)[:, None], cs.shape),
                "pos": cd,
                "rrf": rrf[a:b], "rrf_rank": np.broadcast_to(np.arange(K)[None, :], cs.shape),
                "dn": dn, "dn_gap": rowmax(S["dn"]) - dn,
                "kn": kn, "kn_gap": rowmax(S["kn"]) - kn,
                "bA": bA, "bA_rel": bA / (rowmax(S["bA"]) + 1e-6),
                "bF": bF, "bF_rel": bF / (rowmax(S["bF"]) + 1e-6),
                "covA": g(S["covA"]) / qs.ntok[a:b, None], "covF": g(S["covF"]) / qs.ntok[a:b, None],
                "fcov": fc,
                "rating_ok": np.where(np.isnan(thr), np.nan, (c.rating[cs] >= thr).astype(np.float32)),
                "mc_prob": np.take_along_axis(S["mcp"], c.mc_code[cs], 1),
                "cat_prob": np.take_along_axis(S["ctp"], c.cat_code[cs], 1),
                "cat_match": (c.cat_code[cs] == qcol(qs.cat_code)).astype(np.float32),
                "loc_match": lm.astype(np.float32), "geo_km": geo,
                "nn_sim1": np.broadcast_to(S["nn1"][:, None], cs.shape),
                "q_ntok": qcol(qs.ntok), "q_nchar": qcol(qs.nchar), "q_delivery": qcol(qs.delivery.astype(np.float32)),
                "q_has_filter": qcol(qs.has_filter),
                "price_log": c.price_log[cs], "rating": c.rating[cs], "reviews_log": c.reviews_log[cs],
                "phone_hidden": c.phone_hidden[cs], "msg_forbidden": c.msg_forbidden[cs],
                "title_ntok": c.title_ntok[cs], "desc_ntok": c.desc_ntok[cs],
            }
            parts.append({k: np.asarray(v)[ok] for k, v in f.items()})
            del S
        df = pd.DataFrame({k: np.concatenate([p[k] for p in parts]) for k in parts[0]})
        for k in df.columns:
            if k not in ("qrow", "pos"):
                df[k] = df[k].astype(np.float32)
        keys = df["qrow"].values.astype(np.int64) * c.n + df["pos"].values
        df["is_hist"] = (np.isin(keys, hist_keys).astype(np.float32) if hist_keys is not None and len(hist_keys) else 0.0)
        return df


def fuse(tops, valid, weights, k_rrf, K, N):
    """Взвешенный RRF по всем запросам сразу (векторно). -> cand (nq,K) с -1 в хвосте и rrf-скоры."""
    nq, C, KK = tops.shape
    contrib = (np.asarray(weights, np.float64)[None, :, None] / (k_rrf + np.arange(KK)[None, None, :] + 1.0)) * valid
    keys = (np.arange(nq, dtype=np.int64)[:, None, None] * N + tops).ravel()
    uk, inv = np.unique(keys, return_inverse=True)
    sums = np.bincount(inv, weights=contrib.ravel(), minlength=len(uk))
    keep = sums > 0
    uk, sums = uk[keep], sums[keep]
    q_of, item_of = uk // N, (uk % N).astype(np.int32)
    order = np.lexsort((-sums, q_of))
    q_s, it_s, s_s = q_of[order], item_of[order], sums[order]
    start = np.searchsorted(q_s, np.arange(nq))
    rank = np.arange(len(q_s)) - start[q_s]
    sel = rank < K
    cand = np.full((nq, K), -1, np.int32)
    score = np.zeros((nq, K), np.float32)
    cand[q_s[sel], rank[sel]] = it_s[sel]
    score[q_s[sel], rank[sel]] = s_s[sel]
    return cand, score


def build_gt(uids, pairs, id2pos, N):
    """Разметка «запрос-единица -> позиции релевантных в корпусе» в виде отсортированных ключей row*N+pos."""
    u2r = {u: i for i, u in enumerate(uids)}
    sub = pairs[pairs["uid"].isin(u2r)]
    rows = sub["uid"].map(u2r).values.astype(np.int64)
    pos = sub["item_id"].map(id2pos)
    ok = ~pos.isna().values
    keys = np.unique(rows[ok] * N + pos.values[ok].astype(np.int64))
    n_rel = np.bincount(keys // N, minlength=len(uids))
    return keys, n_rel


def recall_at(cand, gt_keys, n_rel, N, K):
    nq = cand.shape[0]
    c = cand[:, :K]
    rows = np.repeat(np.arange(nq), c.shape[1])
    flat = c.ravel()
    ok = flat >= 0
    hit = np.isin(rows[ok].astype(np.int64) * N + flat[ok], gt_keys)
    per = np.bincount(rows[ok][hit], minlength=nq)
    m = n_rel > 0
    return float(np.mean(per[m] / n_rel[m]))


def tune_rrf(tops, valid, gt_keys, n_rel, N, cfg):
    """Подбор весов RRF по recall кандидатов. Оценка честная: history-канала в валидации нет (запросы не пересекаются)."""
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    Kc, Kf = cfg["k_cand"], cfg["final_k"]
    use = [ci for ci, (_, b, loc) in enumerate(CHANNELS) if b is not None and valid[:, ci].any()]

    def obj(trial):
        w = np.zeros(len(CHANNELS))
        for ci in use:
            w[ci] = trial.suggest_float("w_" + CHANNELS[ci][0], 0.0, 3.0)
        k = trial.suggest_int("k_rrf", 5, 120)
        cand, _ = fuse(tops, valid, w, k, Kc, N)
        return 0.5 * recall_at(cand, gt_keys, n_rel, N, Kf) + 0.5 * recall_at(cand, gt_keys, n_rel, N, Kc)

    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=cfg["seed"]))
    st.enqueue_trial({**{"w_" + CHANNELS[ci][0]: 1.0 for ci in use}, "k_rrf": 60})   # старт: равные веса
    st.optimize(obj, n_trials=cfg["rrf_trials"])
    w = np.zeros(len(CHANNELS))
    for ci in use:
        w[ci] = st.best_params["w_" + CHANNELS[ci][0]]
    w[HIST] = 5.0     # точное совпадение текста запроса + объект есть в корпусе -> сильный сигнал
    log(f"[rrf] best={st.best_value:.4f} params={ {k: round(v, 3) for k, v in st.best_params.items()} }")
    return w, st.best_params["k_rrf"]


In [33]:
# ЯЧЕЙКА 9: реранкер (CatBoost YetiRank) + оценка + важность признаков
def ranked_recall(df, score, n_rel, rows, K):
    """Recall@K после реранкинга. Знаменатель — ВСЕ релевантные единицы запроса (а не только попавшие в кандидаты)."""
    d = pd.DataFrame({"qrow": df["qrow"].values, "s": score, "y": df["label"].values})
    top = d.sort_values(["qrow", "s"], ascending=[True, False]).groupby("qrow", sort=False).head(K)
    per = top.groupby("qrow")["y"].sum().reindex(rows, fill_value=0).values
    nr = n_rel[rows]
    m = nr > 0
    return float(np.mean(per[m] / nr[m]))


def train_ranker(dtr, dva, feats, cfg):
    from catboost import CatBoostRanker, Pool
    ptr = Pool(dtr[feats], dtr["label"], group_id=dtr["qrow"])
    pva = Pool(dva[feats], dva["label"], group_id=dva["qrow"])
    base = dict(loss_function="YetiRank", eval_metric="NDCG:top=50", iterations=cfg["cb_iterations"],
                learning_rate=cfg["cb_lr"], depth=cfg["cb_depth"], random_seed=cfg["seed"],
                verbose=100, early_stopping_rounds=100, use_best_model=True)
    last = None
    for task in ("GPU", "CPU"):                      # на GPU YetiRank в разы быстрее; при ошибке — CPU
        try:
            m = CatBoostRanker(**base, task_type=task)
            m.fit(ptr, eval_set=pva)
            log(f"[ranker] обучен на {task}, best_iter={m.get_best_iteration()}")
            return m
        except Exception as e:
            last = e
            log(f"[ranker] {task} не удался: {str(e)[:150]}")
    raise last


def feature_report(model, X, cfg):
    """v1 падал: для ранжирующих loss тип LossFunctionChange требует Pool. Берём PredictionValuesChange и SHAP через Pool."""
    from catboost import Pool
    imp = pd.DataFrame({"feature": X.columns,
                        "importance": model.get_feature_importance(type="PredictionValuesChange")}
                       ).sort_values("importance", ascending=False)
    imp.to_csv(os.path.join(cfg["work_dir"], "feature_importance.csv"), index=False)
    print("\n[важность признаков, PredictionValuesChange]\n", imp.head(25).to_string(index=False))
    try:
        Xs = X.sample(n=min(cfg["shap_rows"], len(X)), random_state=cfg["seed"])
        sv = model.get_feature_importance(Pool(Xs), type="ShapValues")[:, :-1]
        sh = pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv).mean(0)}
                          ).sort_values("mean_abs_shap", ascending=False)
        sh.to_csv(os.path.join(cfg["work_dir"], "shap_importance.csv"), index=False)
        print("\n[SHAP mean|value|]\n", sh.head(25).to_string(index=False))
    except Exception as e:
        print("[SHAP] пропущен:", str(e)[:200])


In [34]:
# ЯЧЕЙКА 10: проверка answer.csv на соответствие формату задания
def validate_submission(path, bench_q_raw, bench_i_raw):
    a = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(a.columns) == ["query_id", "answer"], "колонки должны быть ровно query_id, answer"
    assert len(a) == len(bench_q_raw) and a["query_id"].is_unique, "должна быть ровно одна строка на query_id"
    assert set(a["query_id"]) == set(bench_q_raw["query_id"].astype(str)), "query_id не совпадают"
    ids = set(bench_i_raw["item_id"].astype(str))
    pat = re.compile(r"^[0-9a-f]{16}$")
    n_empty = 0
    for s in a["answer"]:
        toks = s.split(" ") if s else []
        n_empty += (len(toks) == 0)
        assert len(toks) <= 50 and len(set(toks)) == len(toks), "больше 50 или повторы"
        assert all(pat.match(t) and t in ids for t in toks), "item_id не из корпуса / неверный формат"
    print(f"[validate] OK: {len(a)} строк, пустых ответов: {n_empty}, "
          f"среднее число id: {a['answer'].map(lambda s: len(s.split()) if s else 0).mean():.1f}")


In [35]:
# ЯЧЕЙКА 11: ГЛАВНЫЙ ПАЙПЛАЙН
def run(train_raw, bq_raw, bi_raw, encoder=None, cfg=CFG):
    wd = cfg["work_dir"]; os.makedirs(wd, exist_ok=True)
    rng = np.random.RandomState(cfg["seed"])

    # ---------- 1. препроцессинг и диагностика ----------
    log("[1] препроцессинг")
    pairs, train_items = prep_train(train_raw)
    bench_q, bench_i = prep_queries(bq_raw), prep_items(bi_raw)
    gc.collect()
    loc_eq = diagnostics(pairs, train_items, bench_q, bench_i)
    use_loc = cfg["loc_channel"] if isinstance(cfg["loc_channel"], bool) else loc_eq >= cfg["loc_min_share"]
    log(f"канал «та же локация»: {'ВКЛ' if use_loc else 'ВЫКЛ'}")

    # ---------- 2. разбиение train: дообучение энкодера | отложенные запросы для реранкера ----------
    # Отложенные запросы И их объявления не участвуют в дообучении -> для них энкодер «не видел» ответов,
    # как и для запросов бенчмарка.
    texts = pairs["q_norm"].unique()
    upt = pairs["uid"].nunique() / len(texts)
    n_hold = int(min(cfg["holdout_frac"] * len(texts), cfg["rank_max_units"] / upt))
    hold_texts = set(texts[rng.permutation(len(texts))[:n_hold]])
    is_hold = pairs["q_norm"].isin(hold_texts)
    hold_items = set(pairs.loc[is_hold, "item_id"])
    ft_pairs = pairs[~is_hold & ~pairs["item_id"].isin(hold_items)].drop_duplicates(["q_norm", "item_id"])
    if cfg["ft_max_pairs"] and len(ft_pairs) > cfg["ft_max_pairs"]:
        ft_pairs = ft_pairs.sample(cfg["ft_max_pairs"], random_state=cfg["seed"])
    log(f"[2] отложено текстов: {n_hold}; пар для дообучения: {len(ft_pairs)}")

    # ---------- 3. энкодер: дообучение на кликах ----------
    ft_on = cfg["ft_epochs"] > 0 and encoder is None
    enc = encoder if encoder is not None else Encoder(cfg["model_name"])
    tag = "ft" if ft_on else "base"
    if ft_on:
        ft_dir = os.path.join(wd, "ft_model")
        if cfg["use_cache"] and os.path.exists(ft_dir):
            enc = Encoder(ft_dir); log("[3] загружена дообученная модель из кэша")
        else:
            ti_text = dict(zip(train_items["item_id"], doc_text(train_items)))
            finetune(enc, ft_pairs["q_norm"].tolist(), ft_pairs["item_id"].map(ti_text).tolist(), cfg)
            enc.save(ft_dir)
            del ti_text
    gc.collect()

    def emb(name, txts, prefix, max_len):
        path = os.path.join(wd, f"emb_{tag}_{name}.npy")
        if cfg["use_cache"] and os.path.exists(path):
            x = np.load(path)
            if len(x) == len(txts):
                return x
        x = enc.encode(list(txts), prefix, max_len, cfg["enc_bs"], name)
        if cfg["use_cache"]:
            np.save(path, x)
        return x

    log("[3] эмбеддинги")
    E_bi = emb("bench_items", doc_text(bench_i), D_PREFIX, cfg["max_len_doc"])
    E_ti = emb("train_items", doc_text(train_items), D_PREFIX, cfg["max_len_doc"])
    all_texts = pd.unique(pd.concat([pairs["q_norm"], bench_q["q_norm"]], ignore_index=True))
    E_q = emb("queries", all_texts, Q_PREFIX, cfg["max_len_q"])
    text2row = {t: i for i, t in enumerate(all_texts)}
    log(f"эмбеддинги готовы, RSS={mem():.1f} GB")

    voc = {"cat": Vocab(train_items["item_category_id"], bench_i["item_category_id"],
                        pairs["search_category"], bench_q["search_category"]),
           "mc": Vocab(train_items["item_microcat_id"], bench_i["item_microcat_id"]),
           "loc": Vocab(train_items["item_location_id"], bench_i["item_location_id"],
                        pairs["search_location_id"], bench_q["search_location_id"])}
    tid2row = pd.Series(np.arange(len(train_items)), index=train_items["item_id"].values)
    it_mc, it_cat = voc["mc"].code(train_items["item_microcat_id"]), voc["cat"].code(train_items["item_category_id"])
    g = ft_pairs.groupby("search_location_id")[["item_latitude", "item_longitude"]].median()
    loc_geo = {k: (r["item_latitude"], r["item_longitude"]) for k, r in g.iterrows()}

    def make_pool(p):
        return QueryPool(p, text2row, E_q, tid2row, E_ti, it_mc, it_cat, voc["mc"].n, voc["cat"].n)

    # ---------- 4. валидация на псевдо-корпусе: корпус бенчмарка + объявления отложенных запросов ----------
    log("[4] псевдо-корпус и каналы для отложенных запросов")
    pool_ft = make_pool(ft_pairs)                       # соседи ТОЛЬКО из дообучающей части
    hold_pairs = pairs[is_hold]
    hu = hold_pairs.drop_duplicates("uid")[["uid"] + UNIT_KEYS].reset_index(drop=True)
    bench_ids = set(bench_i["item_id"])
    extra = train_items[train_items["item_id"].isin(set(hold_pairs["item_id"]))
                        & ~train_items["item_id"].isin(bench_ids)]
    pc = Corpus(pd.concat([bench_i, extra], ignore_index=True),
                np.vstack([E_bi, E_ti[extra.index.values]]), voc, "pseudo-corpus")
    gt_keys, n_rel = build_gt(hu["uid"].values, hold_pairs, pc.id2pos, pc.n)
    qs_h = QSet(hu, E_q[[text2row[t] for t in hu["q_norm"]]], voc, loc_geo)
    ret = Retriever(pc, pool_ft, cfg, use_loc)
    tops, valid = ret.channel_tops(qs_h)

    print("\nRecall по каналам на отложенных запросах (псевдо-корпус):")
    for ci, (name, base, _) in enumerate(CHANNELS):
        if base is None or not valid[:, ci].any():
            continue
        lst = np.where(valid[:, ci], tops[:, ci], -1)
        print(f"  {name:10s} R@50={recall_at(lst, gt_keys, n_rel, pc.n, 50):.4f}  "
              f"R@{cfg['k_ch']}={recall_at(lst, gt_keys, n_rel, pc.n, cfg['k_ch']):.4f}")

    # разбиение отложенных единиц на train/val реранкера — по ТЕКСТУ запроса
    tid = pd.factorize(hu["q_norm"])[0]
    is_val = (rng.rand(tid.max() + 1) < 0.2)[tid]
    tr_rows, va_rows = np.where(~is_val)[0], np.where(is_val)[0]

    # веса RRF подбираем на train-части (val остаётся нетронутым)
    sub = tr_rows[:cfg["rrf_tune_units"]]
    gk_s, nr_s = build_gt(hu["uid"].values[sub], hold_pairs, pc.id2pos, pc.n)
    w, k_rrf = tune_rrf(tops[sub], valid[sub], gk_s, nr_s, pc.n, cfg)
    cand, rrf = fuse(tops, valid, w, k_rrf, cfg["k_cand"], pc.n)
    gk_v, nr_v = build_gt(hu["uid"].values[va_rows], hold_pairs, pc.id2pos, pc.n)
    cv_ = cand[va_rows]
    log(f"[RRF, val] Recall@{cfg['final_k']}={recall_at(cv_, gk_v, nr_v, pc.n, cfg['final_k']):.4f}   "
        f"Recall@{cfg['k_cand']} (потолок реранкера)={recall_at(cv_, gk_v, nr_v, pc.n, cfg['k_cand']):.4f}")

    # ---------- 5. реранкер на ТЕХ ЖЕ кандидатах (топ-k_cand), что будут на инференсе ----------
    log("[5] признаки и реранкер")
    df = ret.features(qs_h, cand, rrf)
    df["label"] = np.isin(df["qrow"].values.astype(np.int64) * pc.n + df["pos"].values, gt_keys).astype(np.float32)
    feats = [c for c in df.columns if c not in ("qrow", "pos", "label", "is_hist")]
    dtr, dva = df[~is_val[df["qrow"].values]], df[is_val[df["qrow"].values]]
    log(f"train: {len(dtr)} строк, pos={int(dtr.label.sum())}; val: {len(dva)} строк, pos={int(dva.label.sum())}")
    model = train_ranker(dtr, dva, feats, cfg)
    r50 = ranked_recall(dva, model.predict(dva[feats]), n_rel, va_rows, cfg["final_k"])
    log(f"[РЕАНКЕР, val] Recall@{cfg['final_k']} = {r50:.4f}   "
        f"(RRF без реранкера: {recall_at(cv_, gk_v, nr_v, pc.n, cfg['final_k']):.4f})")
    feature_report(model, dva[feats], cfg)

    # финальный реранкер — на всех отложенных запросах, число деревьев из early stopping
    from catboost import CatBoostRanker, Pool
    best_it = max(50, int(model.get_best_iteration() * 1.1))
    params = {k: v for k, v in model.get_params().items() if k not in ("use_best_model", "early_stopping_rounds", "iterations")}
    final_model = CatBoostRanker(**params, iterations=best_it)
    final_model.fit(Pool(df[feats], df["label"], group_id=df["qrow"]))
    del df, dtr, dva, tops, valid, cand, rrf, pc, ret, qs_h
    gc.collect()

    # ---------- 6. инференс на бенчмарке: корпус = benchmark_items, пул = ВЕСЬ train ----------
    log("[6] инференс")
    pool_all = make_pool(pairs.drop_duplicates(["q_norm", "item_id"]))
    bc = Corpus(bench_i, E_bi, voc, "bench")
    qs_b = QSet(bench_q, E_q[[text2row[t] for t in bench_q["q_norm"]]], voc, loc_geo)

    # history: тот же текст запроса встречался в train И кликнутое объявление есть в корпусе (по id)
    pb = pairs[pairs["item_id"].isin(bc.id2pos)]
    cnt = pb.groupby(["q_norm", "item_id"]).size().reset_index(name="n").sort_values("n", ascending=False)
    hist = cnt.groupby("q_norm")["item_id"].apply(lambda s: [bc.id2pos[i] for i in s][:cfg["hist_max"]]).to_dict()
    hist_lists = [hist.get(t, []) for t in bench_q["q_norm"]]
    log(f"запросов бенчмарка с history-объявлениями в корпусе: {sum(1 for h in hist_lists if h)} из {len(hist_lists)}")
    hist_keys = np.array([i * bc.n + p for i, h in enumerate(hist_lists) for p in h], np.int64)

    ret_b = Retriever(bc, pool_all, cfg, use_loc)
    tops_b, valid_b = ret_b.channel_tops(qs_b, hist_lists)
    cand_b, rrf_b = fuse(tops_b, valid_b, w, k_rrf, cfg["k_cand"], bc.n)
    db = ret_b.features(qs_b, cand_b, rrf_b, hist_keys)
    db["score"] = final_model.predict(db[feats]) + 1000.0 * db["is_hist"].values   # history-объявления — вперёд
    top = db.sort_values(["qrow", "score"], ascending=[True, False]).groupby("qrow", sort=False).head(cfg["final_k"])
    lists = top.groupby("qrow")["pos"].apply(list).to_dict()
    ans = [" ".join(bc.ids[lists.get(i, [])]) for i in range(qs_b.n)]
    out = pd.DataFrame({"query_id": bench_q["query_id"].values, "answer": ans})
    out.to_csv(cfg["output_csv"], index=False)
    log(f"[6] записан {cfg['output_csv']}")
    validate_submission(cfg["output_csv"], bq_raw, bi_raw)
    return out


In [36]:
# ЯЧЕЙКА 12: ЗАГРУЗКА И ЗАПУСК
# Стандартное чтение локальных parquet-файлов через pandas.
# Ожидаются файлы train.parquet, benchmark_queries.parquet и benchmark_items.parquet
# в текущей рабочей директории (или соответствующие пути из CFG).
train_raw = pd.read_parquet(CFG["train_path"])
bq_raw = pd.read_parquet(CFG["bench_queries_path"])
bi_raw = pd.read_parquet(CFG["bench_items_path"])

print("train_raw:", train_raw.shape)
print("bq_raw:", bq_raw.shape)
print("bi_raw:", bi_raw.shape)
print("First 5 train records:", train_raw.head())

answer = run(train_raw, bq_raw, bi_raw)


[     16s] [1] препроцессинг
[     28s] ======================================================================
[     28s] ДИАГНОСТИКА ДАННЫХ
строк(кликов)=467029, уникальных текстов=73873, уникальных «запросов-единиц»=354305, кликов на единицу: mean=1.32, median=1
train-объявлений всего=344825; из них есть в корпусе бенчмарка: 18142 (5.3%). Если ~0% — history по item_id почти бесполезен, а train нужен для обучения по ТЕКСТАМ объявлений.
доля запросов бенчмарка, чей текст встречается в train: 37.4%

search_category в train (топ): {'114': 466991, '0': 34, '81': 2, '10': 2}
search_category в бенчмарке (топ): {'114': 2230, '0': 222}
item_category_id в корпусе: unique = 47 | топ: {'114': 187336, '19': 357, '112': 141, '40': 122, '33': 103, '10': 98}
кросс-таб train: search_category -> item_category_id (топ-3 на строку):
   0: {'114': 34, '10': 0, '112': 0}
   10: {'114': 2, '10': 0, '112': 0}
   114: {'114': 466977, '27': 3, '19': 2}
   81: {'114': 2, '10': 0, '112': 0}

доля кликов с item_

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[     33s] [encoder] intfloat/multilingual-e5-small on cuda


finetune epoch 1/5:   0%|          | 0/1564 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 192.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 110.81 MiB is free. Including non-PyTorch memory, this process has 14.45 GiB memory in use. Of the allocated memory 13.70 GiB is allocated by PyTorch, and 631.35 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)